# Food Delivery Data — Cleaning & KPI Analysis

**Tools:** Python (pandas)  \
**Goal:** Take a raw, messy food-delivery dataset (~14,700 orders) through a full cleaning pipeline, then calculate business KPIs.

> This is a practice project on a synthetic (sample) dataset built to demonstrate a real cleaning + analysis workflow.

**Approach:** diagnose every problem first, then clean in a deliberate order, then analyse.
The cleaning covers the six most common data-quality issues: missing values, duplicates,
impossible values, inconsistent text, wrong data types, and mixed formats.


## 1. Load the data


In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('Food_Data_messy.csv')
df.head()

## 2. Diagnose — measure every problem before changing anything

Golden rule: never clean what you haven't measured. First we profile the whole dataset.


In [ ]:
# Data types + non-null counts. Any 'object' column that should be a number/date is a red flag,
# and any non-null count below the total row count means missing values.
df.info()

In [ ]:
# Missing values per column
df.isna().sum()

In [ ]:
# Duplicates: full-row copies vs repeated OrderIDs (an OrderID should be unique)
print('Full-row duplicates:', df.duplicated().sum())
print('Duplicate OrderIDs :', df['OrderID'].duplicated().sum())

In [ ]:
# Numeric overview. Watch DeliveryTimeMin: a negative min and a 999 max signal impossible values,
# and a mean well above the median signals those bad values inflating the average.
df.describe()

In [ ]:
# Impossible values in DeliveryTimeMin: <=0 (hard errors) and 999 (a 'sentinel' for unknown)
print('Values <= 0 :', (df['DeliveryTimeMin'] <= 0).sum())
print('Value == 999:', (df['DeliveryTimeMin'] == 999).sum())

In [ ]:
# Text inconsistency check (casing, spaces, abbreviations). Example: City
df['City'].value_counts(dropna=False)

### Findings
- **City:** 300 blanks + casing/space/abbreviation mess (`Sthlm`, `Gbg`)
- **DeliveryStatus:** 220 blanks + variants (`On Time`, `Ontime`, casing)
- **CustomerType / Cuisine:** casing + whitespace variants
- **OrderValueSEK:** stored as text (`248 kr`, `270,0`) — should be a number
- **OrderDate:** stored as text in mixed formats — should be a datetime
- **DeliveryTimeMin:** negatives, zeros, and a `999` sentinel
- **Duplicates:** many repeated OrderIDs, mostly disguised by formatting

**Cleaning order matters:** standardize text and fix types *before* removing duplicates,
otherwise formatting differences (`Stockholm` vs `stockholm `) hide the duplicates.


## 3. Clean — text columns

Two stages per text column: an **automatic sweep** (`.str.strip().str.title()`) to fix casing and
whitespace, then a **manual map** (`.replace`) for abbreviations the sweep can't handle.


In [ ]:
# City: sweep, then map abbreviations
df['City'] = df['City'].str.strip().str.title()
df['City'] = df['City'].replace({'Sthlm': 'Stockholm', 'Gbg': 'Göteborg'})
df['City'].value_counts(dropna=False)

In [ ]:
# Cuisine: sweep only (no abbreviations)
df['Cuisine'] = df['Cuisine'].str.strip().str.title()
df['Cuisine'].value_counts(dropna=False)

In [ ]:
# DeliveryStatus: sweep, then map ('.title()' turns 'on time' into 'On Time'; 'Ontime' has no space)
df['DeliveryStatus'] = df['DeliveryStatus'].str.strip().str.title()
df['DeliveryStatus'] = df['DeliveryStatus'].replace({'On Time': 'On time', 'Ontime': 'On time'})
df['DeliveryStatus'].value_counts(dropna=False)

In [ ]:
# CustomerType: sweep only
df['CustomerType'] = df['CustomerType'].str.strip().str.title()
df['CustomerType'].value_counts(dropna=False)

## 4. Clean — fix data types

`OrderValueSEK` is text because of currency words and comma-decimals; `OrderDate` is text in mixed formats.


In [ ]:
# OrderValueSEK -> number. Clean the text until every cell is a valid number-string, then convert.
df['OrderValueSEK'] = (df['OrderValueSEK']
                       .str.replace('kr', '', regex=False)
                       .str.replace('SEK', '', regex=False)
                       .str.replace(' ', '', regex=False))   # remove currency words + spaces
df['OrderValueSEK'] = df['OrderValueSEK'].str.replace(',', '.', regex=False)  # Swedish comma -> dot
df['OrderValueSEK'] = pd.to_numeric(df['OrderValueSEK'])                      # text -> float
df['OrderValueSEK'].dtype

In [ ]:
# OrderDate -> datetime. format='mixed' handles the different formats; dayfirst=True reads DD/MM.
df['OrderDate'] = pd.to_datetime(df['OrderDate'], format='mixed', dayfirst=True)
df['OrderDate'].dtype

## 5. Clean — impossible values

Turn the impossible delivery times into blanks (NaN) — keep the row's good data, blank only the bad cell.


In [ ]:
df.loc[df['DeliveryTimeMin'] <= 0, 'DeliveryTimeMin'] = np.nan   # negatives and zeros
df.loc[df['DeliveryTimeMin'] == 999, 'DeliveryTimeMin'] = np.nan # the 999 sentinel
df['DeliveryTimeMin'].describe()   # mean is now realistic (no longer inflated by 999s)

## 6. Clean — missing values

Decide per column: **median** for the numeric column (robust to outliers), **"Unknown"** for text
categories (keeps the gap visible and countable).


In [ ]:
median_time = df['DeliveryTimeMin'].median()          # store it so it recalculates on new data
df['DeliveryTimeMin'] = df['DeliveryTimeMin'].fillna(median_time)

df['City'] = df['City'].fillna('Unknown')
df['DeliveryStatus'] = df['DeliveryStatus'].fillna('Unknown')

df.isna().sum()   # should all be 0

## 7. Clean — duplicates (last)

Now that text and types are standardized, the disguised duplicates are visible and can be removed.


In [ ]:
print('Duplicates now visible:', df.duplicated().sum())   # much higher than at the start
df = df.drop_duplicates()                                 # keeps first copy, drops the rest
print('Duplicates remaining :', df.duplicated().sum())
print('Final shape          :', df.shape)

## 8. Save the cleaned dataset


In [ ]:
df.to_csv('Food_orders_clean.csv', index=False)
print('Saved Food_orders_clean.csv')

## 9. KPIs

With clean data, the KPIs are trustworthy. Whole-dataset numbers use `.sum()`/`.mean()`;
per-category numbers use `groupby`.


In [ ]:
# Total revenue
total_revenue = df['OrderValueSEK'].sum()
print('Total revenue (SEK):', round(total_revenue))

In [ ]:
# Average delivery time
print('Average delivery time (min):', round(df['DeliveryTimeMin'].mean(), 1))

In [ ]:
# Top 3 cities by revenue
df.groupby('City')['OrderValueSEK'].sum().sort_values(ascending=False).head(3)

In [ ]:
# Top 3 cuisines by revenue
df.groupby('Cuisine')['OrderValueSEK'].sum().sort_values(ascending=False).head(3)

In [ ]:
# Average delivery time per cuisine
df.groupby('Cuisine')['DeliveryTimeMin'].mean().round(1)

In [ ]:
# Percentage of late deliveries (mean of a True/False column = the proportion that are True)
late_pct = (df['DeliveryStatus'] == 'Late').mean() * 100
print('Late deliveries (%):', round(late_pct, 1))

## 10. Reusable cleaning pipeline

All the cleaning steps above, wrapped in a single function so next month's file can be cleaned in one line
instead of re-running every cell. Fill values (like the median) are recalculated inside the function,
so the pipeline adapts to each new dataset. `errors='coerce'` keeps an unexpected value from crashing the run.

This is the *reusability* goal: raw file in -> clean DataFrame out.


In [ ]:
def clean_orders(filepath):
    """Take a raw orders CSV and return a cleaned DataFrame. Reusable for any month's file."""
    df = pd.read_csv(filepath)

    # --- text columns: sweep (casing/whitespace) + map (abbreviations) ---
    df['City'] = df['City'].str.strip().str.title()
    df['City'] = df['City'].replace({'Sthlm': 'Stockholm', 'Gbg': 'Göteborg'})
    df['Cuisine'] = df['Cuisine'].str.strip().str.title()
    df['DeliveryStatus'] = df['DeliveryStatus'].str.strip().str.title()
    df['DeliveryStatus'] = df['DeliveryStatus'].replace({'On Time': 'On time', 'Ontime': 'On time'})
    df['CustomerType'] = df['CustomerType'].str.strip().str.title()

    # --- data types ---
    df['OrderValueSEK'] = (df['OrderValueSEK']
                           .str.replace('kr', '', regex=False)
                           .str.replace('SEK', '', regex=False)
                           .str.replace(' ', '', regex=False)
                           .str.replace(',', '.', regex=False))
    df['OrderValueSEK'] = pd.to_numeric(df['OrderValueSEK'], errors='coerce')
    df['OrderDate'] = pd.to_datetime(df['OrderDate'], format='mixed', dayfirst=True)

    # --- impossible values -> blank ---
    df.loc[df['DeliveryTimeMin'] <= 0, 'DeliveryTimeMin'] = np.nan
    df.loc[df['DeliveryTimeMin'] == 999, 'DeliveryTimeMin'] = np.nan

    # --- missing values (recalculated each run, so it adapts to new data) ---
    df['DeliveryTimeMin'] = df['DeliveryTimeMin'].fillna(df['DeliveryTimeMin'].median())
    df['City'] = df['City'].fillna('Unknown')
    df['DeliveryStatus'] = df['DeliveryStatus'].fillna('Unknown')

    # --- duplicates last (after standardizing, so disguised copies are visible) ---
    df = df.drop_duplicates()

    return df

In [ ]:
# Clean any month's file in one line:
clean_df = clean_orders('Food_Data_messy.csv')

# Quick validation that it worked
print('Shape       :', clean_df.shape)
print('Nulls       :', clean_df.isna().sum().sum())        # expect 0
print('Duplicates  :', clean_df.duplicated().sum())        # expect 0
print('Value dtype :', clean_df['OrderValueSEK'].dtype)    # float64
print('Date dtype  :', clean_df['OrderDate'].dtype)        # datetime

## Key takeaways
- Data must be standardized (text + types) **before** duplicates can be reliably detected.
- Cleaning choices (median vs mean, "Unknown" vs blank) should be deliberate and explainable.
- Storing fill values in variables and avoiding fixed ranges makes the workflow repeatable on next month's data.
